# MEIDNet in 5 minutes

This notebook runs the **published perovskite model** on a CPU: pick a band-gap target, let MEIDNet search its latent space, and read the report that explains every candidate.

Nothing here is specific to perovskites except the *family file*; the same commands work on your own data (see notebook 02).

In [ ]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu
!pip -q install git+https://github.com/ABnano/MEIDNet.git


## 1. Choose a target
Band gap in eV and formation enthalpy in eV/atom (the two properties the published model knows). The report flags targets outside what the model saw in training.

In [ ]:
family = 'halide'      #@param ['halide', 'oxide', 'chalcogenide', 'nitride']
band_gap = 2.0         #@param {type:'number'}
enthalpy = -0.10       #@param {type:'number'}
n_candidates = 3       #@param {type:'integer'}


## 2. Run the search
`meidnet demo` downloads the 2.8 MB checkpoint, searches, applies the family's rules and writes CIFs + a report.

In [ ]:
!meidnet demo --family {family} --band-gap {band_gap} --enthalpy {enthalpy} -n {n_candidates} --rounds 4 --steps 200 --no-open --device cpu

## 3. Read the report
The verdict, the *funnel* (where attempts were rejected), and one card per candidate with its checklist and predicted-vs-target gauges.

In [ ]:
from IPython.display import HTML, display
display(HTML(open('runs/demo/generation_report.html', encoding='utf-8').read()))

## 4. Look at a structure

In [ ]:
import glob
from pymatgen.core import Structure
for f in sorted(glob.glob('runs/demo/generation/cifs/*.cif')):
    s = Structure.from_file(f)
    print(f.split('/')[-1], s.composition.reduced_formula, f'a = {s.lattice.a:.3f} Å', s.get_space_group_info())

## 5. The whole design space at once
For a prototype family every possible composition can be listed. `meidnet space` scores all of them with the structure encoder — compare with what the latent search found.

In [ ]:
!meidnet init --template perov5 -o perov5.yaml --force
!meidnet space perov5.yaml --model ~/.meidnet/dual_autoencoder_clip_earlyfusion_propertyaware_2k.pth -o space.csv
import pandas as pd
df = pd.read_csv('space.csv')
df[df.passes_all].assign(dist=(df.pred_dir_gap-band_gap).abs()).sort_values('dist').head(10)

## Next
* Notebook **02** — the same pipeline on *your* table of structures and properties.
* `meidnet studio` locally — change rules and targets and watch the effect live.
* [Documentation](https://abnano.github.io/MEIDNet)